# Fase 2: Preprocesamiento e Ingeniería de Variables (Versión Corregida)
## Sistema de Predicción de Factores de Riesgo en Adolescentes Salvadoreños (GSHS 2013)

**Propósito:** Este notebook toma el dataset limpio de la fase anterior y prepara las matrices para los modelos. 
**NOTA CRÍTICA (CORRECCIÓN):** A diferencia de la versión anterior, **hemos eliminado el escalado y la codificación global** de este notebook. 
¿Por qué? Aplicar `RobustScaler` o `OneHotEncoder` a todo el conjunto de datos antes de dividir en Entrenamiento/Validación/Prueba introduce **Fuga de Datos (Data Leakage)**, ya que las estadísticas de la prueba se filtran al entrenamiento.

**Nueva Arquitectura:** 
1.  Cargamos los datos.
2.  Seleccionamos las características (excluyendo `QN`, `Q4`, `Q5`, `Q25`, etc.).
3.  Convertimos las columnas predictoras a tipo `string` (para que el `Pipeline` las reconozca como categóricas).
4.  Dividimos los datos en 70/20/10.
5.  Guardamos los datos **crudos** (sin escalar y sin codificar).

El escalado y la codificación se realizarán de forma segura dentro del `Pipeline` en la siguiente fase.

In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split

print("Librerías de preprocesamiento cargadas exitosamente.")

Librerías de preprocesamiento cargadas exitosamente.


## 1. Carga de Datos y Separación de Variables (Q vs QN)

**¿Qué hace este bloque?**
Carga los datos procesados previamente. Luego, separa las variables predictoras ($X$) de las variables objetivo ($y$). Además, elimina las columnas redundantes y aquellas que causan *Data Leakage*.

**¿Por qué lo hacemos así?**
El dataset original contiene preguntas categóricas (`Q1` a `Q58`) y sus recodificaciones numéricas (`QN6` a `QN58`). Utilizar ambas simultáneamente generaría una **multicolinealidad perfecta**, por lo que eliminamos las `QN`.
Además, **es obligatorio** eliminar el Peso (`Q5`) y la Estatura (`Q4`) del conjunto de variables predictoras al predecir el IMC, así como la pregunta base del riesgo de salud mental (`Q25`).

In [2]:
# Carga del dataset limpio
df = pd.read_csv('../data/processed/SLV2013_Limpios_Targets.csv')

# Definición de las variables objetivo (Targets)
y_reg = df['IMC']
y_clf = df['Riesgo_Salud_Mental']

# Filtramos las columnas 'QN' y variables derivadas globales (suelen empezar con 'qn' minúscula)
columnas_a_eliminar = [col for col in df.columns if col.startswith('QN') or col.startswith('qn')]

# Agregamos las variables que causan Data Leakage y los Targets al filtro de eliminación
columnas_a_eliminar.extend(['Q4', 'Q5', 'Q25', 'IMC', 'Riesgo_Salud_Mental'])

# Variables auxiliares del diseño de la encuesta que no son predictoras
columnas_a_eliminar.extend(['weight', 'stratum', 'psu'])

# Matriz de características (Features) - Solo las variables Q restantes
X = df.drop(columns=columnas_a_eliminar, errors='ignore')

print(f"Dimensiones de X (Predictoras crudas): {X.shape}")
print(f"Dimensiones de y_reg (Target Regresión): {y_reg.shape}")
print(f"Dimensiones de y_clf (Target Clasificación): {y_clf.shape}")

Dimensiones de X (Predictoras crudas): (1915, 46)
Dimensiones de y_reg (Target Regresión): (1915,)
Dimensiones de y_clf (Target Clasificación): (1915,)


## 2. Preparación para el Pipeline (Conversión a String)

**¿Qué hace este bloque?**
Convierte todas las columnas predictoras a tipo de dato `string` (cadena de texto).

**¿Por qué lo hacemos así?**
Nuestro `build_preprocessor()` en `pipeline_factory.py` utiliza `make_column_selector(dtype_include=['object', 'category', 'bool'])` para identificar automáticamente las variables categóricas y aplicarles el `OneHotEncoder`. Dado que nuestras variables `Q` son numéricas en el CSV (ej. 1, 2, 3), si no las convertimos a string, el selector las tratará como numéricas y no se codificarán correctamente. Al convertirlas a string aquí, aseguramos que el pipeline ejecute el flujo categórico correcto durante el entrenamiento.

In [3]:
# Convertimos todas las columnas de X a tipo string 
# para que el Pipeline las detecte como categóricas en el Notebook 03
X = X.astype(str)

print("Variables predictoras convertidas a tipo string. Listas para la división.")

Variables predictoras convertidas a tipo string. Listas para la división.


In [4]:
from sklearn.impute import SimpleImputer

# --- IMPUTACIÓN DE TARGETS ---
# El IMC (Regresión) debe imputarse con la mediana para preservar la distribución 
# y evitar NaN en el entrenamiento de RandomForest.
imputador_imc = SimpleImputer(strategy='median')
y_reg = pd.Series(imputador_imc.fit_transform(y_reg.values.reshape(-1, 1)).flatten(), name='IMC')

# El Riesgo de Salud Mental (Clasificación) lo creamos con un mapeo (1 o 0), 
# pero por seguridad, rellenamos posibles NaN con 0 (Sin Riesgo).
# (Si bien el Notebook 1 ya lo hace, esta línea es una doble seguridad).
y_clf = y_clf.fillna(0)

print("Targets imputados exitosamente. Ya no contienen NaN.")

Targets imputados exitosamente. Ya no contienen NaN.


## 3. División de Datos (70/20/10) y Exportación

**¿Qué hace este bloque?**
Guarda las matrices finales $X$ (crudas y en string), $y\\_reg$ y $y\\_clf$ en la carpeta de procesados. 
**Importante:** Aquí NO se aplica ni One-Hot Encoding ni Escalado. El pipeline del Notebook 03 se encargará de eso sobre la marcha, garantizando cero fugas de datos.

**¿Por qué lo hacemos así?**
Aislar este paso permite que el Notebook 03 cargue particiones fijas y estables, lo que facilita la experimentación y la comparación de modelos sin tener que recalcular la división de datos cada vez.

In [5]:
from sklearn.impute import SimpleImputer

# --- IMPUTACIÓN DE TARGETS (ANTES DE LA DIVISIÓN) ---
# El IMC (Regresión) se imputa con la mediana para preservar la distribución.
imputador_imc = SimpleImputer(strategy='median')
y_reg = pd.Series(imputador_imc.fit_transform(y_reg.values.reshape(-1, 1)).flatten(), name='IMC')

# El Riesgo de Salud Mental se llena con 0 (Sin Riesgo) por si acaso hay algún nulo.
y_clf = y_clf.fillna(0)

print("Targets imputados exitosamente. Ya no contienen NaN.")

Targets imputados exitosamente. Ya no contienen NaN.


In [6]:
# 1. Primer corte: 70% Entrenamiento y 30% Temporal (para Validación y Pruebas)
X_train, X_temp, y_reg_train, y_reg_temp, y_clf_train, y_clf_temp = train_test_split(
    X, y_reg, y_clf, 
    test_size=0.30, 
    random_state=42 # Semilla para reproducibilidad
)

# 2. Segundo corte: Del 30% temporal, asignamos 1/3 a Pruebas (10% del total) 
# y 2/3 a Validación (20% del total)
X_val, X_test, y_reg_val, y_reg_test, y_clf_val, y_clf_test = train_test_split(
    X_temp, y_reg_temp, y_clf_temp, 
    test_size=1/3, # 1/3 de 30% = 10%
    random_state=42
)

# 3. Guardado de las matrices separadas en disco de forma ordenada
# Features (X) - Crudas y en String
X_train.to_csv('../data/processed/X_train.csv', index=False)
X_val.to_csv('../data/processed/X_val.csv', index=False)
X_test.to_csv('../data/processed/X_test.csv', index=False)

# Target Regresión (y_reg)
y_reg_train.to_csv('../data/processed/y_reg_train.csv', index=False)
y_reg_val.to_csv('../data/processed/y_reg_val.csv', index=False)
y_reg_test.to_csv('../data/processed/y_reg_test.csv', index=False)

# Target Clasificación (y_clf)
y_clf_train.to_csv('../data/processed/y_clf_train.csv', index=False)
y_clf_val.to_csv('../data/processed/y_clf_val.csv', index=False)
y_clf_test.to_csv('../data/processed/y_clf_test.csv', index=False)

# 4. Verificación de las dimensiones
print("División 70/20/10 completada con éxito.")
print(f"Entrenamiento: {X_train.shape[0]} registros (Aprox 70%)")
print(f"Validación: {X_val.shape[0]} registros (Aprox 20%)")
print(f"Pruebas: {X_test.shape[0]} registros (Aprox 10%)")
print("Matrices exportadas en ../data/processed/. Listas para la fase de Modelado.")

División 70/20/10 completada con éxito.
Entrenamiento: 1340 registros (Aprox 70%)
Validación: 383 registros (Aprox 20%)
Pruebas: 192 registros (Aprox 10%)
Matrices exportadas en ../data/processed/. Listas para la fase de Modelado.
